# Color-based pipeline overview

Run the implemented pipeline prefix: board detection -> scoring-board and
number-ring normalization -> printed-20 orientation -> geometry-defined
field interiors -> four robust per-image color models -> dart-likelihood.
This is **not** an end-to-end scorer: entry points and scores remain unknown.

The reference template is board-style specific. A covered `20` can give a
wrong uncertain estimate; the planned `5`/`1` fallback is not implemented.
Batch mode below shows final outputs only; return to the selected-image
cells to investigate any bad filename. Nothing is automatically saved.

In [ ]:
import json
from functools import partial
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from dartcv import (
    Printed20OrientationStage, extract_printed20_template,
    format_timing_summary, normalize_board_view, prepare_board,
)
from dartcv.color_dart_inspection import plot_orientation

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
IMAGE_DIR = ROOT / 'data/images/multi_dart'
MANIFEST = json.loads((ROOT / 'data/annotations/color_dart_orientation.json').read_text())
PHOTO_PATHS = sorted(IMAGE_DIR.glob('*.jpg'))
if not PHOTO_PATHS:
    raise FileNotFoundError('Restore data/images using the README DVC workflow first.')
NORMALIZER = partial(normalize_board_view, number_ring_radius_mm=240.0)

def read_photo(name):
    image = cv2.imread(str(IMAGE_DIR / name))
    if image is None:
        raise FileNotFoundError(IMAGE_DIR / name)
    return image

def source_label_angle(preparation, source_xy):
    point = np.linalg.inv(preparation.board.model_to_source) @ np.array([*source_xy, 1.0])
    point = point[:2] / point[2]
    return float(np.rad2deg(np.arctan2(point[0], -point[1])))


## Prepare the explicit reference template once

The manifest stores approximate manually inspected source-image label centers, not angles predicted by the matcher. Projection into the detected board model converts that independent observation to an angle. No source imagery is embedded in this notebook. The annular crop spans radii 202–228 mm and 14 degrees; inspect it before trusting the matcher.

In [ ]:
reference = MANIFEST['reference']
reference_preparation = prepare_board(
    read_photo(reference['image']), board_normalization=NORMALIZER,
)
if reference_preparation.failure is not None:
    raise RuntimeError(reference_preparation.failure)
reference_angle = source_label_angle(reference_preparation, reference['label_center_source_px'])
template = extract_printed20_template(
    reference_preparation.board_view, label_angle_deg=reference_angle,
)
ORIENTATION_STAGE = Printed20OrientationStage(template)
plt.figure(figsize=(4, 3))
plt.imshow(cv2.cvtColor(template, cv2.COLOR_BGR2RGB))
plt.title('Reference 20 — inspect this template')
plt.axis('off')
plt.show()
print(format_timing_summary(reference_preparation))


In [ ]:
from dartcv import analyze_board_colors, RobustBoardColorSegmentationStage
from dartcv.color_dart_inspection import (
    plot_board_preparation, plot_orientation, plot_color_segmentation,
)

SEGMENTATION_STAGE = RobustBoardColorSegmentationStage(
    sample_margin_mm=2.5, wire_margin_mm=1.0,
    minimum_component_area_mm2=3.0,
)
IMAGE_INDEX = 1
image_path = PHOTO_PATHS[IMAGE_INDEX]
analysis = analyze_board_colors(
    read_photo(image_path.name), orientation=ORIENTATION_STAGE,
    segmentation=SEGMENTATION_STAGE, diagnostics=True,
)
print(image_path.name)
for step in analysis.steps:
    print(f"{step.name}: {step.status.value}, {step.elapsed_ms:.2f} ms")
    if step.failure:
        print("  FAILED:", step.failure)
    for substep, milliseconds in step.substep_timings_ms.items():
        print(f"  {substep}: {milliseconds:.2f} ms")

## Board preparation and orientation

Inspect geometry before judging segmentation. The color-pattern
homography alone does not verify numbered sectors.

In [ ]:
display(plot_board_preparation(analysis.preparation))
if analysis.orientation is not None and analysis.orientation.artifacts:
    display(plot_orientation(analysis.orientation))

## Field masks, color models and anomaly output

Samples are pooled by expected field color, excluding nominal wire margins
and missing source pixels. Hue wraps continuously in value-weighted HSV
cone coordinates; dark pixels carry little hue weight. Median/MAD fitting
trims sparse contamination. Inspect centers, robust scales and retained
support below; they are not global fixed color thresholds.

Raw anomaly strength is a heuristic 0–1 score, not a calibrated probability.
Cleanup now seeds components outside wire/sampling margins, then restores
raw wire-crossing evidence within a finite 3 mm reach. It does not open
away thin shafts or recursively flood the grid. The old hard-grid/opening
policy remains available for comparison. Some reflections still remain;
raw evidence is retained for later tip work.
Incorrect geometry/orientation and majority-contaminated samples can spoil
these models. No exactly-three-blob assumption or automatic score is made.

A weak numbered-label match no longer rotates the color-field masks: they retain the detector's paint alignment (still ambiguous modulo 36 degrees). Numbered orientation and eventual scores remain uncertain. This assumes the supplied detector homography is paint-aligned.


In [ ]:
if analysis.segmentation is None:
    print("Segmentation skipped: inspect the earlier failure above.")
elif not analysis.segmentation.artifacts:
    print("Segmentation unavailable:", analysis.segmentation.failure)
else:
    display(plot_color_segmentation(analysis.segmentation))
    artifacts = analysis.segmentation.artifacts
    if "color_model_centers" in artifacts:
        for index, color in enumerate(artifacts["color_model_names"]):
            print(color, "center", artifacts["color_model_centers"][index],
                  "scale", artifacts["color_model_scales"][index],
                  "sampled/retained", artifacts["color_model_support"][index])

## Investigating a bad batch tile

Set `IMAGE_INDEX` to the named photo (or select `image_path` explicitly),
rerun analysis, and inspect the figures above. Use the dedicated
`color_dart_board_inspection.ipynb` / `color_dart_orientation_inspection.ipynb`
for deeper geometry/number evidence. Uncertain orientation remains visible
even when a color map was produced. Next: entry-point localization, then
independent scoring; neither is fabricated here.

## Compact batch: final outputs only

Set `RUN_BATCH = True` and edit `BATCH_PATHS` to select any subset (remove `[:3]`
for the full directory). Run this cell again without repeating the intermediate
visualizations. Processing is sequential; pages retain small output thumbnails,
not full-resolution diagnostic histories. Each tile names the input, final
status and processing time. Failures are visible, not silently skipped.

To investigate a bad tile, copy its filename into this notebook's single-image
selection and rerun the detailed sections. Batch mode uses automatic calibration;
never reuse one image's manual points/tips on another image.

Run reference-template and segmentation setup once first. Batches use the same analysis helper and stage objects as the detailed cells.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from dartcv.notebook_batch import BatchOutput, image_batch_figures, read_batch_image

BATCH_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())

RUN_BATCH = False
BATCH_PATHS = sorted((BATCH_ROOT / "data/images/multi_dart").glob("*.jpg"))[:3]

from dartcv import analyze_board_colors
from dartcv.color_dart_inspection import likelihood_overlay

def batch_output(path):
    analysis = analyze_board_colors(read_batch_image(path), orientation=ORIENTATION_STAGE,
                                    segmentation=SEGMENTATION_STAGE, diagnostics=False)
    if analysis.preparation.failure is not None:
        return BatchOutput(None, f"FAILED: {analysis.preparation.failure.message}")
    if analysis.orientation.value is None:
        return BatchOutput(None, f"Orientation FAILED: {analysis.orientation.failure}")
    if analysis.segmentation.value is None:
        return BatchOutput(None, f"Segmentation FAILED: {analysis.segmentation.failure}")
    estimate = analysis.orientation.value
    status = "orientation confident" if estimate.confident else "UNCERTAIN orientation"
    view = analysis.preparation.board_view
    return BatchOutput(likelihood_overlay(view.image, analysis.segmentation.value.probabilities),
                       f"Color anomalies; {status}; tips/scores unknown")

if RUN_BATCH:
    for batch_figure in image_batch_figures(BATCH_PATHS, batch_output, page_size=6):
        display(batch_figure)
        plt.close(batch_figure)

## Compare color distributions and grid cleanup

Enable the optional cell below for the currently selected image. Rows compare
expected-field matching, matching against any of the four field-trained colors,
and a four-mode distribution fitted to **all observed board pixels**, without
field assignments. Columns compare no cleanup, the old hard grid mask/opening,
and the new support-preserving cleanup. Geometry/orientation are identical.

Whole-board matching can hide white/metal dart pixels resembling other board
colors. Hard grid removal can erase real shafts. These are experimental
alternatives, not automatic accuracy guarantees. For a selectable batch and
sparse manual-probe evaluation, open `color_dart_segmentation_comparison.ipynb`.

In [ ]:
from dartcv.color_dart_evaluation import compare_segmentations, plot_segmentation_comparison

RUN_ABLATION = False
if RUN_ABLATION:
    if analysis.orientation is None or analysis.orientation.value is None:
        raise RuntimeError("A supported orientation estimate is required; inspect the earlier failure.")
    variants = compare_segmentations(
        analysis.preparation.board_view, analysis.preparation.board,
        analysis.orientation.value,
    )
    display(plot_segmentation_comparison(analysis.preparation.board_view, variants))